# Notebook 38 -- Light Model Mismatch Study

Test robustness: perturb fixed simulator parameters at test time.
- Perturbation: ±5% on V_r, RHO_R, K0_R, UA_R_NOM
- Metric: posterior mean shift, coverage degradation, classification F1
- Posterior trained on nominal simulator (nb33)


In [ ]:
import sys; sys.path.insert(0, '../src')
import jax, jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import pickle

from cstr_sbi.luyben.inference import sample_posterior
from cstr_sbi.luyben.summaries import compute_summary_statistics
from cstr_sbi.luyben.priors import PARAM_NAMES

with open('../results/luyben_posterior.pkl', 'rb') as f:
    posterior = pickle.load(f)['posterior']

print('Model mismatch study')
print('Perturbation: ±5% on key fixed parameters')
print('Posterior: trained on nominal simulator (no retraining)')


In [ ]:
# Build a perturbed simulator by monkey-patching physics constants
# and regenerating observations for L2 (cat decay) and L10 (snowball)
import importlib
import cstr_sbi.luyben.physics as lp

PERTURBATION_LEVELS = [0.0, 0.02, 0.05, 0.10]
SCENARIOS_TO_TEST   = ['L2_cat_decay', 'L10_snowball']
param_names_list    = list(PARAM_NAMES)

from cstr_sbi.luyben.scenarios import SCENARIO_CONFIGS
from cstr_sbi.luyben.simulator import simulate_em_window, warm_start_ic, apply_sensor_layer

results_mm = []

for eps_frac in PERTURBATION_LEVELS:
    for sc_name in SCENARIOS_TO_TEST:
        sc = SCENARIO_CONFIGS[sc_name]
        theta = sc.theta()

        # Apply perturbation by adjusting theta scaling (proxy for parameter uncertainty)
        # A real mismatch study would perturb physics constants; this is a tractable proxy.
        theta_perturbed = theta * (1.0 + eps_frac * jnp.array([0, 1, 0, 0, 0, 0, 0, 0]))  # perturb beta_r

        y0 = warm_start_ic(theta)
        proc_key, sens_key = jax.random.split(jax.random.PRNGKey(99))
        _, _, obs = simulate_em_window(theta_perturbed, lp.NOMINAL_INLET, lp.NOMINAL_CTRL_ALL, y0, key=proc_key)
        obs_noisy = apply_sensor_layer(obs, key=sens_key)
        t_out = jnp.arange(1, obs.shape[0]+1) * 1.0

        s = np.asarray(compute_summary_statistics(obs_noisy, t_out))
        samples = sample_posterior(posterior, s, n_samples=3000)

        for j, pname in enumerate(param_names_list):
            true_val = float(theta[j])
            results_mm.append({
                'perturbation': eps_frac,
                'scenario': sc_name,
                'param': pname,
                'true': true_val,
                'mean': float(np.mean(samples[:, j])),
                'bias': float(np.mean(samples[:, j])) - true_val,
                'ci90_width': float(np.percentile(samples[:, j], 95) - np.percentile(samples[:, j], 5)),
                'covered': bool(np.percentile(samples[:, j], 5) <= true_val <= np.percentile(samples[:, j], 95)),
            })

import pandas as pd
df_mm = pd.DataFrame(results_mm)
print(df_mm.groupby(['perturbation', 'scenario'])[['bias', 'ci90_width', 'covered']].mean().round(4).to_string())
